# ProtCell inference of unmeasured proteins

Adapt ProtCell to the measured panel, query five additional identities from its protein vocabulary, and inspect the inferred cytotoxic program and measured-protein support.

Run the cells in order with the `protein_llm` kernel. Edit the data and checkpoint paths in the setup cell. The `RUN_*` switches enable the corresponding fine-tuning, prediction or analysis steps.

In [ ]:
from pathlib import Path
import os
import sys
import tempfile

PROJECT_ROOT = Path(os.environ.get("PROTCELL_PROJECT_ROOT", str(Path.cwd()))).expanduser().resolve()
if not (PROJECT_ROOT / "train.py").is_file() and (PROJECT_ROOT.parent / "train.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "train.py").is_file(), "Set PROTCELL_PROJECT_ROOT to your ProtCell clone before starting JupyterLab."
OUTPUT_ROOT = Path(os.environ.get("PROTCELL_OUTPUT_DIR", str(PROJECT_ROOT / "results/outputs")))
OUTPUT_DIR = OUTPUT_ROOT / "unobserved"
FIGURE_DIR = OUTPUT_DIR / "figures"
RESULT_DIR = OUTPUT_DIR / "results"
MODEL_DIR = OUTPUT_DIR / "models"
for directory in [OUTPUT_DIR, FIGURE_DIR, RESULT_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.chdir(OUTPUT_DIR) # trainer logs and new files stay with this experiment

old_cache_settings = {key: os.environ.get(key) for key in ["MPLCONFIGDIR", "NUMBA_CACHE_DIR", "XDG_CACHE_HOME"]}
plot_cache = tempfile.TemporaryDirectory(prefix="protcell-unobserved-", dir=OUTPUT_DIR)
os.environ["MPLCONFIGDIR"] = plot_cache.name
os.environ["NUMBA_CACHE_DIR"] = str(Path(plot_cache.name) / "numba")
os.environ["XDG_CACHE_HOME"] = str(Path(plot_cache.name) / "xdg")

import json
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse, stats
from tqdm.auto import tqdm
from IPython.display import display
from model.model import ProteinLanguageModel
from typing import List
from finetuner.Imputation import ImputationTrainer
from statsmodels.stats.multitest import multipletests
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "font.family": "sans-serif"})
sc.settings.figdir = FIGURE_DIR
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DATA_ROOT = Path(os.environ.get("PROTCELL_DATA_ROOT", str(PROJECT_ROOT / "data"))).expanduser().resolve()
BASE_MODEL_PATH = PROJECT_ROOT / "checkpoints/basemodel/large-scale-new-10/model_369999.pt"
print("Device:", DEVICE)
print("Outputs:", OUTPUT_DIR)
from typing import Sequence

## Load target cells and define the unmeasured proteins

Batch 1 was used for adaptation, and batch 2 contains 24,972 evaluation cells from ten biological samples.

In [ ]:
full_data = ad.read_h5ad(DATA_ROOT / "CLR_data/Dataset83.h5ad")
train_data = full_data[full_data.obs.batch.astype(str).eq("1")].copy()
adata = full_data[full_data.obs.batch.astype(str).eq("2")].copy()
TARGET_PROTEINS = ["GZMB", "PRF1", "KLRD1", "TBX21", "EOMES"]
HIGH_CELLTYPES = ["NK cells", "Unconventional T cells", "CD8+ Memory T"]
assert not set(TARGET_PROTEINS).intersection(adata.var_names)
assert set(train_data.obs_names).isdisjoint(adata.obs_names)
assert adata.shape == (24972, 79) and adata.obs["sample"].nunique() == 10
display(adata)
PAPER_CHECKPOINT_PATH = PROJECT_ROOT / "checkpoints/imputation/83_batch1/base_10.pt"

FINETUNED_MODEL_DIR = MODEL_DIR / "masked_protein"
FINETUNED_MODEL_PATH = FINETUNED_MODEL_DIR / "83_batch1/base_10.pt"
IMPUTATION_MODEL_PATH = PAPER_CHECKPOINT_PATH # Or FINETUNED_MODEL_PATH to reuse a new adaptation.
RUN_FINETUNING = True
RUN_INFERENCE = True
assert RUN_INFERENCE, "Enable RUN_INFERENCE before running the analysis."

# Plot/data conventions from results/unobserved_imputation.ipynb.
SAMPLE_COLUMN = "sample"
CELLTYPE_COLUMN = "celltype"
SUBTYPE_COLUMN = "subtype"
HIGH_GROUP = "High cytotoxic types"
OTHER_GROUP = "Other types"
HIGH_STATUS = "Higher in high-cytotoxic types"
OTHER_STATUS = "Higher in other types"
FDR_CUTOFF = 0.05
EFFECT_CUTOFF = 0.5
TOP_N_EACH = 5
PALETTE = {HIGH_STATUS: "#0B77B5", OTHER_STATUS: "#E69F00", "NS": "#C9CED6"}
CELLTYPE_ORDER = ["CD4+ Naive T", "CD8+ Naive T (some DNT)", "CD4+ Memory T", "B cells", "pDC",
                 "Non-classical monocytes", "Classical Monocytes and mDC", "Unconventional T cells",
                 "CD8+ Memory T", "NK cells"]


## Optional masked-protein adaptation

Use batch 1 for adaptation and batch 2 for inference. `train_cls=False` optimizes the masked protein-position output used by `predict_values` below.

New adaptation weights are saved under `models/masked_protein/83_batch1/`; the following loading cell uses that checkpoint. Set `RUN_FINETUNING=False` to use the configured existing checkpoint. In either case, predictions and plots are calculated in this run.


In [ ]:
if RUN_FINETUNING:
    base_model = ProteinLanguageModel.load_model_from_checkpoint(str(BASE_MODEL_PATH), device=DEVICE).to(DEVICE)
    trainer = ImputationTrainer(model=base_model, adata=train_data, device=DEVICE, batch_size=512,
                                expr_name="83_batch1", train_cls=False, lr=1e-3, masked_ratio=0.15, seed=SEED,
                                trainable_modules={"lora": 1e-3, "reg_proj": 1e-5, "expr_head": 5e-5})
    trainer.fit(num_epochs=10, save=True, save_dir=str(FINETUNED_MODEL_DIR), save_epoch_interval=10,
                lora_rank=16, lora_alpha=64, lora_dropout=0.1)
    IMPUTATION_MODEL_PATH = FINETUNED_MODEL_PATH
    assert IMPUTATION_MODEL_PATH.is_file(), "Fine-tuning did not produce the expected checkpoint."
    print("Saved masked-protein model:", IMPUTATION_MODEL_PATH)

## Load ProtCell and inspect the codebook

The queried identities are absent from this panel but must already exist in the pretrained protein vocabulary.

In [ ]:
import hashlib

IMPUTATION_MODEL_PATH = Path(IMPUTATION_MODEL_PATH).resolve()
if RUN_FINETUNING:
    assert IMPUTATION_MODEL_PATH == FINETUNED_MODEL_PATH.resolve(), "Run the fine-tuning cell first, or set RUN_FINETUNING=False."
assert IMPUTATION_MODEL_PATH.is_file(), IMPUTATION_MODEL_PATH
model = ProteinLanguageModel.load_model_from_checkpoint(str(IMPUTATION_MODEL_PATH), device=DEVICE).to(DEVICE)
model.eval()
LOADED_MODEL_PATH = IMPUTATION_MODEL_PATH
LOADED_MODEL_SHA256 = hashlib.sha256(LOADED_MODEL_PATH.read_bytes()).hexdigest()
assert set(TARGET_PROTEINS).issubset(model.tokenizer.protein_to_id)
print("Inference checkpoint:", LOADED_MODEL_PATH)
print("Checkpoint SHA256:", LOADED_MODEL_SHA256)
print("Prediction head: masked protein-position output (expr_preds)")
display(model)
display(pd.Series({protein: model.tokenizer.protein_to_id[protein] for protein in TARGET_PROTEINS}, name="token_id"))

## Masked forward pass

The existing basic inference function adds missing query tokens and masks their abundances.

In [ ]:
def predict_values(model, 
                        adata, 
                        predict_protein: List,
                        device='cuda',
                        batch_size=2048,
                        show_tqdm = True,
                        ):

    assert len(predict_protein) > 0, "predict_protein cannot be empty"
    assert len(set(predict_protein)) == len(predict_protein), \
        "predict_protein has duplicate protein names"
    assert adata.var_names.is_unique, "adata.var_names has duplicate protein names"

    model.eval()
    model.to(device)

    # generate mask and add protein
    missing = [p for p in predict_protein if p not in adata.var_names]

    X = adata.X.toarray() if sparse.issparse(adata.X) else adata.X

    if len(missing) > 0:
        print(f"补充缺失的蛋白列: {missing}")
        added_values = np.zeros(shape=(X.shape[0], len(missing)))

        X = np.hstack((X, added_values))

        var_old = adata.var.copy()
        var_new = pd.DataFrame(index=missing)
        var_final = pd.concat([var_old, var_new])

        adata = ad.AnnData(
            X=X,
            obs=adata.obs.copy(),
            var=var_final
        )

    input_ids = np.array(model.tokenizer.encode(adata.var_names.to_list()))
    input_ids = torch.tensor(input_ids).to(device)
    input_ids = input_ids.expand(adata.n_obs, -1)
    expr_values = torch.tensor(X).float().to(device)

    var_names = np.array(adata.var_names.tolist())
    mask_local = np.isin(var_names, np.array(predict_protein))

    # 模型按矩阵列顺序返回被 mask 的蛋白，
    # 记录如何将输出重排为 predict_protein 的顺序。
    masked_proteins = var_names[mask_local].tolist()
    output_order = [masked_proteins.index(p) for p in predict_protein]

    expr_preds_all = []
    a = tqdm(range(0, adata.n_obs, batch_size), desc="Predict Protein Values") if show_tqdm else range(0, adata.n_obs, batch_size)
    for i in a:
        batch_expr = expr_values[i : i + batch_size, :].float()
        batch_ids = input_ids[i : i + batch_size, :]
        B, L = batch_expr.shape
        mask = np.tile(mask_local, (B, 1))
        mask = torch.tensor(mask, dtype=torch.bool, device=device)

        with torch.no_grad():
            expr_preds, _ = model(input_ids=batch_ids, expr_values=batch_expr, mask=mask)
            expr_preds_all.append(expr_preds.cpu())

    predictions = (
        torch.cat(expr_preds_all, dim=0)
        .reshape(adata.n_obs, len(masked_proteins))
        .numpy()
    )

    return predictions[:, output_order]

## Infer abundances for the target cells

Run inference and export predictions with checkpoint provenance. Check that the loaded checkpoint matches the selected path and has not changed since loading. Fig. 2f, g and i use these predictions; Fig. 2h uses measured abundances.


In [ ]:
if RUN_INFERENCE:
    assert LOADED_MODEL_PATH == Path(IMPUTATION_MODEL_PATH).resolve(), "Checkpoint selection changed: rerun the model-loading cell."
    assert hashlib.sha256(LOADED_MODEL_PATH.read_bytes()).hexdigest() == LOADED_MODEL_SHA256, "Checkpoint was overwritten after loading: rerun the model-loading cell."
    print("Predicting with:", LOADED_MODEL_PATH)
    inferred = predict_values(model, adata, TARGET_PROTEINS, device=DEVICE, batch_size=512)
    predictions = pd.DataFrame(inferred, index=adata.obs_names, columns=TARGET_PROTEINS)
    per_cell = adata.obs[["sample", "celltype", "subtype"]].join(predictions.add_prefix("pred_"))
    per_cell.to_csv(RESULT_DIR / "new_per_cell_predictions.csv.gz", index_label="cell_id", compression="gzip")
    prediction_version = "new"
else:
    raise AssertionError("Enable RUN_INFERENCE to calculate predictions; saved results are not loaded.")
display(per_cell.head())

prediction_run_info = {
    "dataset": "Dataset83", "training_batch": "1", "target_batch": "2",
    "n_cells": adata.n_obs, "n_samples": adata.obs[SAMPLE_COLUMN].nunique(),
    "targets": TARGET_PROTEINS, "prediction_version": prediction_version,
    "inference_performed": RUN_INFERENCE, "finetuning_performed": RUN_FINETUNING,
    "checkpoint": str(LOADED_MODEL_PATH),
    "checkpoint_sha256": LOADED_MODEL_SHA256,
    "prediction_head": "masked protein-position output",
    "prediction_source": str(RESULT_DIR / "new_per_cell_predictions.csv.gz"),
}
(RESULT_DIR / "Fig2fghi_run_info.json").write_text(json.dumps(prediction_run_info, indent=2) + "\n")
display(pd.Series(prediction_run_info, name="prediction_provenance"))

## Figure helpers
Define helpers for matrix conversion, standardization and figure export.

In [ ]:
def dense_array(matrix):
    return matrix.toarray() if sparse.issparse(matrix) else np.asarray(matrix)

def zscore_columns(frame):
    scale = frame.std(axis=0, ddof=0).replace(0, np.nan)
    result = (frame - frame.mean(axis=0)) / scale
    if result.isna().any().any():
        bad = result.columns[result.isna().any()].tolist()
        raise ValueError(f"Cannot z-score constant or missing predictions: {bad}")
    return result

def save_figure(figure, stem):
    figure.savefig(FIGURE_DIR / f"{stem}.png", dpi=600, bbox_inches="tight")
    figure.savefig(FIGURE_DIR / f"{stem}.pdf", bbox_inches="tight")

## Fig. 2f — Inferred cytotoxic program

Calculate the mean of the five standardized predictions per cell. Draw the cell-level score violin using the fixed cell-type order, tab10 colours, quartile lines and a zero reference.


In [ ]:
prediction_z = zscore_columns(predictions)

score_table = adata.obs[[SAMPLE_COLUMN, CELLTYPE_COLUMN]].copy()
score_table["Predicted cytotoxic score"] = prediction_z.mean(axis=1)

celltype_order = CELLTYPE_ORDER

# 更鲜艳的离散配色
palette = sns.color_palette("tab10", n_colors=len(celltype_order))


# ============================================================
# Violin plot
# ============================================================

fig, ax = plt.subplots(figsize=(11.5, 6.8))

sns.violinplot(
    data=score_table,
    x=CELLTYPE_COLUMN,
    y="Predicted cytotoxic score",

    order=celltype_order,
    palette=palette,

    inner="quartile",
    cut=0,

    # 稍微增强轮廓
    linewidth=0.9,

    # 每个 violin 使用相同最大宽度
    density_norm="width",

    # 稍微收窄 violin，避免相邻类别显得拥挤
    width=0.78,

    ax=ax,
)


# ============================================================
# Reference line
# ============================================================

ax.axhline(
    0,
    color="#666666",
    linestyle="--",
    linewidth=1.8,
    zorder=1,
)

# ============================================================
# Labels
# ============================================================

ax.set_title(
    "Inferred cytotoxic program",
    loc="left",
    fontweight="bold",
    fontsize=17,
    pad=10,
)

ax.set_xlabel("")

ax.set_ylabel(
    "Predicted cytotoxic score",
    fontsize=15,
)


# ============================================================
# X-axis labels
# ============================================================

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=45,
    ha="right",
    va="top",
    fontsize=12.5,
)

ax.tick_params(
    axis="y",
    labelsize=12,
)

ax.tick_params(
    axis="x",
    pad=7,
)


# ============================================================
# Grid & spines
# ============================================================

ax.grid(
    axis="y",
    color="#E8E8E8",
    linewidth=0.7,
)

ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_linewidth(0.9)
ax.spines["bottom"].set_linewidth(0.9)


# ============================================================
# Layout
# ============================================================

# 不完全依赖 tight_layout，
# 给下面竖直的 cell-type labels 留足空间
fig.subplots_adjust(
    left=0.11,
    right=0.98,
    top=0.90,
    bottom=0.34,
)


save_figure(
    fig,
    "Fig2f_predicted_cytotoxic_score_violin"
)

plt.show()


# ============================================================
# Save underlying data
# ============================================================

score_table.to_csv(
    RESULT_DIR / "per_cell_predicted_cytotoxic_score.csv.gz",
    compression="gzip",
)

score_summary = (
    score_table
    .groupby(
        CELLTYPE_COLUMN,
        observed=True
    )["Predicted cytotoxic score"]
    .agg(
        n_cells="size",
        mean="mean",
        median="median",
        std="std",
    )
    .sort_values(
        "median",
        ascending=False
    )
)

score_summary.to_csv(
    RESULT_DIR / "predicted_cytotoxic_score_by_celltype.csv"
)

display(score_summary)

## Fig. 2g — Inferred proteins across cell types

Compute per-sample cell-type means from the same standardized predictions used for f and i, then give each biological sample equal weight.

In [ ]:
# Heatmap of the five unmeasured cytotoxic proteins across cell types
heatmap_cells = pd.concat([
    adata.obs[[SAMPLE_COLUMN, CELLTYPE_COLUMN]],
    prediction_z[TARGET_PROTEINS],
], axis=1)

# Give each biological sample equal weight within each cell type.
sample_celltype_means = (
    heatmap_cells
    .groupby([SAMPLE_COLUMN, CELLTYPE_COLUMN], observed=True)[TARGET_PROTEINS]
    .mean()
    .reset_index()
)
celltype_protein_heatmap = (
    sample_celltype_means
    .groupby(CELLTYPE_COLUMN, observed=True)[TARGET_PROTEINS]
    .mean()
    .T
    .loc[TARGET_PROTEINS, celltype_order]
)
assert np.isfinite(celltype_protein_heatmap.to_numpy()).all()


celltype_protein_heatmap.to_csv(RESULT_DIR / f"{prediction_version}_calculated_heatmap.csv", index_label="imputed_protein")
celltype_protein_heatmap.to_csv(RESULT_DIR / "Fig2g_heatmap_source.csv", index_label="imputed_protein")

heatmap_limit = float(np.abs(celltype_protein_heatmap.to_numpy()).max())
assert heatmap_limit > 0, "Heatmap values are unexpectedly constant"
fig, ax = plt.subplots(figsize=(10.2, 4.2))
sns.heatmap(
    celltype_protein_heatmap,
    cmap="vlag",
    center=0,
    vmin=-heatmap_limit,
    vmax=heatmap_limit,
    annot=True,
    fmt=".2f",
    annot_kws={"fontsize": 8},
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Mean predicted expression (z-score)", "shrink": 0.82},
    ax=ax,
)
ax.set_title(
    "Inferred cytotoxic-associated proteins across cell types",
    loc="left", fontweight="bold", fontsize=12,
)
ax.set_xlabel("Cell type", fontsize=10)
ax.set_ylabel("Unmeasured protein", fontsize=10)
ax.set_xticklabels(
    ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor", fontsize=9,
)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)
colorbar = ax.collections[0].colorbar
colorbar.ax.tick_params(labelsize=8)
colorbar.set_label("Mean predicted expression (z-score)", fontsize=9)
fig.tight_layout()
save_figure(fig, "Fig2g_imputed_cytotoxic_proteins_by_celltype_heatmap")
plt.show()
display(celltype_protein_heatmap)

## Paired-sample statistics and volcano function

Calculate paired sample-level protein differences and visualize standardized CLR effects with paired t-test FDR.

In [ ]:
def build_sample_group_means(expression, metadata):
    if len(expression) != len(metadata):
        raise ValueError("Expression and metadata must have the same rows")
    table = metadata[[SAMPLE_COLUMN, CELLTYPE_COLUMN]].reset_index(drop=True).copy()
    table["cytotoxic_group"] = np.where(
        table[CELLTYPE_COLUMN].astype(str).isin(HIGH_CELLTYPES),
        HIGH_GROUP,
        OTHER_GROUP,
    )
    table = pd.concat([table, expression.reset_index(drop=True)], axis=1)
    means = table.groupby(
        [SAMPLE_COLUMN, "cytotoxic_group"], observed=True
    )[expression.columns.tolist()].mean().reset_index()
    group_counts = means.groupby(SAMPLE_COLUMN, observed=True)["cytotoxic_group"].nunique()
    if not group_counts.eq(2).all():
        raise ValueError("Every biological sample must contain both comparison groups")
    return means


def compute_paired_differential(sample_group_means, cell_expression):
    rows = []
    for protein in cell_expression.columns:
        paired = sample_group_means.pivot(
            index=SAMPLE_COLUMN, columns="cytotoxic_group", values=protein
        ).dropna()
        high = paired[HIGH_GROUP].to_numpy(dtype=float)
        other = paired[OTHER_GROUP].to_numpy(dtype=float)
        delta = high - other
        ttest_pvalue = float(stats.ttest_rel(high, other, nan_policy="omit").pvalue)
        try:
            wilcoxon_pvalue = float(stats.wilcoxon(delta, zero_method="wilcox").pvalue)
        except ValueError:
            wilcoxon_pvalue = 1.0
        protein_sd = float(cell_expression[protein].std(ddof=1))
        standardized_delta = float(delta.mean() / protein_sd) if protein_sd > 0 else 0.0
        rows.append({
            "protein": protein,
            "n_samples": len(delta),
            "mean_high": high.mean(),
            "mean_other": other.mean(),
            "mean_delta": delta.mean(),
            "median_delta": np.median(delta),
            "standardized_delta": standardized_delta,
            "positive_samples": int((delta > 0).sum()),
            "negative_samples": int((delta < 0).sum()),
            "ttest_pvalue": ttest_pvalue if np.isfinite(ttest_pvalue) else 1.0,
            "wilcoxon_pvalue": wilcoxon_pvalue if np.isfinite(wilcoxon_pvalue) else 1.0,
        })
    result = pd.DataFrame(rows)
    result["ttest_fdr"] = multipletests(result["ttest_pvalue"], method="fdr_bh")[1]
    result["wilcoxon_fdr"] = multipletests(result["wilcoxon_pvalue"], method="fdr_bh")[1]
    result["status"] = "NS"
    significant = result["ttest_fdr"].lt(FDR_CUTOFF)
    result.loc[significant & result["standardized_delta"].ge(EFFECT_CUTOFF), "status"] = HIGH_STATUS
    result.loc[significant & result["standardized_delta"].le(-EFFECT_CUTOFF), "status"] = OTHER_STATUS
    return result.sort_values("standardized_delta", ascending=False).reset_index(drop=True)

In [ ]:
def _require_columns(frame: pd.DataFrame, columns: Sequence[str], name: str) -> None:
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")

def _set_plot_style() -> None:
    plt.rcParams.update(
        {
            "font.family": "Arial",
            "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans", "sans-serif"],
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "font.size": 8,
            "axes.titlesize": 11,
            "axes.labelsize": 8,
            "xtick.labelsize": 7,
            "ytick.labelsize": 7,
            "legend.fontsize": 7,
            "axes.linewidth": 0.7,
        }
    )

def _save_figure(figure: plt.Figure, output_prefix: str | Path) -> None:
    output_prefix = Path(output_prefix)
    output_prefix.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(output_prefix.with_suffix(".png"), dpi=600, bbox_inches="tight")
    figure.savefig(output_prefix.with_suffix(".pdf"), bbox_inches="tight")

def _select_directional_top(
    differential: pd.DataFrame,
    top_n_each: int,
) -> pd.DataFrame:
    if top_n_each < 1:
        raise ValueError("top_n_each must be positive")
    high = (
        differential.loc[differential["status"].eq(HIGH_STATUS)]
        .nlargest(top_n_each, "standardized_delta")
    )
    other = (
        differential.loc[differential["status"].eq(OTHER_STATUS)]
        .nsmallest(top_n_each, "standardized_delta")
    )
    return (
        pd.concat([other, high], ignore_index=True)
        .sort_values("standardized_delta")
        .reset_index(drop=True)
    )

def plot_protein_volcano(
    differential: pd.DataFrame,
    output_prefix: str | Path,
    top_n_each: int = 4,
    effect_cutoff: float = 0.5,
    fdr_cutoff: float = 0.05,
) -> tuple[plt.Figure, plt.Axes, pd.DataFrame]:
    """Plot sample-aware measured-protein effects and paired-test FDR values."""

    _require_columns(
        differential,
        ["protein", "standardized_delta", "ttest_fdr", "status"],
        "differential table",
    )
    plot_data = differential.copy()
    positive_fdr = plot_data.loc[plot_data["ttest_fdr"].gt(0), "ttest_fdr"]
    floor = float(positive_fdr.min() * 0.1) if len(positive_fdr) else 1e-300
    plot_data["fdr_for_plot"] = plot_data["ttest_fdr"].clip(lower=floor)
    plot_data["minus_log10_fdr"] = -np.log10(plot_data["fdr_for_plot"])
    labels = _select_directional_top(plot_data, top_n_each)

    _set_plot_style()
    figure, axis = plt.subplots(figsize=(5.4, 4.1))
    for status in ["NS", OTHER_STATUS, HIGH_STATUS]:
        subset = plot_data.loc[plot_data["status"].eq(status)]
        axis.scatter(
            subset["standardized_delta"],
            subset["minus_log10_fdr"],
            s=24 if status == "NS" else 38,
            c=PALETTE[status],
            alpha=0.45 if status == "NS" else 0.88,
            edgecolors="white",
            linewidths=0.45,
            label=status,
            zorder=2 if status == "NS" else 3,
        )
    axis.axvline(effect_cutoff, color="#777777", linestyle="--", linewidth=0.8)
    axis.axvline(-effect_cutoff, color="#777777", linestyle="--", linewidth=0.8)
    axis.axhline(-np.log10(fdr_cutoff), color="#777777", linestyle="--", linewidth=0.8)
    texts = []
    for _, row in labels.iterrows():
        texts.append(
            axis.text(
                row["standardized_delta"],
                row["minus_log10_fdr"],
                row["protein"],
                fontsize=6.5,
                fontweight="bold",
                color="#333333",
            )
        )
    try:
        from adjustText import adjust_text

        adjust_text(
            texts,
            ax=axis,
            arrowprops={"arrowstyle": "-", "color": "#999999", "lw": 0.5},
            expand_points=(1.25, 1.35),
            expand_text=(1.20, 1.30),
            ensure_inside_axes=True,
        )
    except ImportError:
        pass
    axis.set_xlabel("Standardized mean CLR difference (high - other)")
    axis.set_ylabel("-log10 paired-sample FDR")
    axis.set_title(
        "Measured-protein differential analysis",
        loc="left",
        fontweight="bold",
    )
    axis.grid(color="#EEEEEE", linewidth=0.55)
    axis.spines[["top", "right"]].set_visible(False)
    axis.legend(
        frameon=False,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.18),
        ncol=3,
        columnspacing=1.2,
        handletextpad=0.4,
    )
    figure.text(
        0.12,
        0.93,
        "Dataset83 batch 2; 10 paired biological samples; all 79 measured proteins",
        fontsize=7.5,
        color="#666666",
        ha="left",
    )
    figure.subplots_adjust(left=0.14, right=0.98, bottom=0.27, top=0.84)
    _save_figure(figure, output_prefix)
    return figure, axis, plot_data

## Fig. 2h — Paired measured-protein differential analysis

Keep the ten biological samples as the paired statistical units. Paired t-test FDR supplies the volcano axis; paired Wilcoxon results are retained in the table.

In [ ]:
measured_expression = pd.DataFrame(dense_array(adata.X).astype(float), index=adata.obs_names, columns=adata.var_names)
metadata = adata.obs[[SAMPLE_COLUMN, CELLTYPE_COLUMN]].copy()
sample_group_means = build_sample_group_means(measured_expression, metadata)
differential = compute_paired_differential(sample_group_means, measured_expression)
assert differential.n_samples.eq(10).all()
differential.to_csv(RESULT_DIR / "measured_protein_differential.csv", index=False)
sample_group_means.to_csv(RESULT_DIR / "measured_sample_group_means.csv", index=False)
display(differential[["protein", "standardized_delta", "ttest_fdr", "wilcoxon_fdr", "status"]])

## Fig. 2h — Standard volcano plot

Call the existing plot_protein_volcano, including blue/orange directional colours, five labels per direction, effect/FDR thresholds, and its annotation layout.

In [ ]:
fig, ax, volcano_data = plot_protein_volcano(
    differential,
    output_prefix=FIGURE_DIR / "Fig2h_measured_protein_volcano",
    top_n_each=TOP_N_EACH,
    effect_cutoff=EFFECT_CUTOFF,
    fdr_cutoff=FDR_CUTOFF,
)
volcano_data.to_csv(RESULT_DIR / "Fig2h_volcano_source.csv", index=False)
plt.show()


## Fig. 2i — NK subtype data and paired-sample contrasts

Use all CD16++ NK and CD56hi/CD16lo NK cells. Compute per-sample medians, paired differences and five-protein FDR before plotting. No cell subsampling is introduced.

In [ ]:
# NK subtype protein-resolution analysis: fixed checkpoint, all cells, paired samples
NK_SUBTYPES = ["CD16++ NK", "CD56hi/CD16lo NK"]
NK_COLORS = {"CD16++ NK": "#E69F00", "CD56hi/CD16lo NK": "#7B61A8"}
DELTA_LABEL = "CD16++ − CD56hi/CD16lo"
z_columns = [f"z_{target}" for target in TARGET_PROTEINS]

nk_protein_z = pd.concat([
    adata.obs[[SAMPLE_COLUMN, CELLTYPE_COLUMN, SUBTYPE_COLUMN]].copy(),
    prediction_z.add_prefix("z_"),
], axis=1)
nk_protein_z = nk_protein_z[
    nk_protein_z[CELLTYPE_COLUMN].eq("NK cells")
    & nk_protein_z[SUBTYPE_COLUMN].isin(NK_SUBTYPES)
].copy()
assert set(nk_protein_z[SUBTYPE_COLUMN]) == set(NK_SUBTYPES)

nk_sample_medians = (
    nk_protein_z.groupby([SAMPLE_COLUMN, SUBTYPE_COLUMN], observed=True)[z_columns]
    .median()
    .reset_index()
)
samples_per_subtype = nk_sample_medians.groupby(SUBTYPE_COLUMN, observed=True)[
    SAMPLE_COLUMN
].nunique()
assert samples_per_subtype.eq(adata.obs[SAMPLE_COLUMN].nunique()).all()
nk_sample_medians.to_csv(
    RESULT_DIR / "nk_subtype_sample_protein_medians.csv", index=False
)

contrast_rows = []
sample_differences = {}
for target in TARGET_PROTEINS:
    column = f"z_{target}"
    paired = nk_sample_medians.pivot(
        index=SAMPLE_COLUMN, columns=SUBTYPE_COLUMN, values=column
    )[NK_SUBTYPES].dropna()
    delta = paired[NK_SUBTYPES[0]] - paired[NK_SUBTYPES[1]]
    sample_differences[target] = delta
    try:
        wilcoxon_p = stats.wilcoxon(delta, alternative="two-sided").pvalue
    except ValueError:
        wilcoxon_p = np.nan
    contrast_rows.append({
        "protein": target,
        "contrast": DELTA_LABEL,
        "n_samples": len(delta),
        "n_cells_cd16pp": int(nk_protein_z[SUBTYPE_COLUMN].eq(NK_SUBTYPES[0]).sum()),
        "n_cells_cd56hi_cd16lo": int(nk_protein_z[SUBTYPE_COLUMN].eq(NK_SUBTYPES[1]).sum()),
        "cd16pp_pooled_median_z": nk_protein_z.loc[
            nk_protein_z[SUBTYPE_COLUMN].eq(NK_SUBTYPES[0]), column
        ].median(),
        "cd56hi_cd16lo_pooled_median_z": nk_protein_z.loc[
            nk_protein_z[SUBTYPE_COLUMN].eq(NK_SUBTYPES[1]), column
        ].median(),
        "sample_delta_mean": delta.mean(),
        "sample_delta_median": delta.median(),
        "sample_delta_q25": delta.quantile(0.25),
        "sample_delta_q75": delta.quantile(0.75),
        "positive_samples": int(delta.gt(0).sum()),
        "paired_t_pvalue": stats.ttest_1samp(delta, 0).pvalue,
        "paired_wilcoxon_pvalue": wilcoxon_p,
    })

nk_contrasts = pd.DataFrame(contrast_rows)
nk_contrasts["paired_t_fdr"] = multipletests(
    nk_contrasts["paired_t_pvalue"], method="fdr_bh"
)[1]
nk_contrasts.to_csv(RESULT_DIR / "nk_subtype_protein_contrasts.csv", index=False)

delta_matrix = pd.DataFrame(sample_differences).T.loc[TARGET_PROTEINS]
delta_matrix.index.name = "protein"
delta_matrix.to_csv(RESULT_DIR / "nk_subtype_paired_delta_matrix.csv")
delta_display = delta_matrix.copy()
delta_display["Median"] = delta_matrix.median(axis=1)

profile_long = nk_protein_z[[SUBTYPE_COLUMN] + z_columns].melt(
    id_vars=SUBTYPE_COLUMN, value_vars=z_columns,
    var_name="protein", value_name="inferred_z",
)
profile_long["protein"] = profile_long["protein"].str.removeprefix("z_")


display(nk_contrasts)

## Fig. 2i — Split-violin NK protein profiles

Display the horizontal five-protein split violins, orange/purple subtype colours, pooled median and interquartile overlays, zero line.

In [ ]:
fig, ax_distribution = plt.subplots(figsize=(5.4, 4.2))
ax_distribution.axhspan(-0.5, 1.5, color="#FFF7E6", zorder=0)
ax_distribution.axhspan(1.5, 4.5, color="#F3F6FA", zorder=0)
sns.violinplot(
    data=profile_long, y="protein", x="inferred_z",
    hue=SUBTYPE_COLUMN, order=TARGET_PROTEINS, hue_order=NK_SUBTYPES,
    split=True, inner=None, cut=0, density_norm="width", gap=0.04,
    palette=NK_COLORS, linewidth=0.8, saturation=0.90, ax=ax_distribution,
)
subtype_offsets = {NK_SUBTYPES[0]: -0.10, NK_SUBTYPES[1]: 0.10}
for y, target in enumerate(TARGET_PROTEINS):
    for subtype in NK_SUBTYPES:
        values = nk_protein_z.loc[
            nk_protein_z[SUBTYPE_COLUMN].eq(subtype), f"z_{target}"
        ]
        q25, median, q75 = values.quantile([0.25, 0.50, 0.75])
        y_offset = y + subtype_offsets[subtype]
        ax_distribution.plot(
            [q25, q75], [y_offset, y_offset], color="#333333",
            linewidth=2.2, solid_capstyle="round", zorder=5,
        )
        ax_distribution.scatter(
            median, y_offset, s=38, color=NK_COLORS[subtype],
            edgecolor="white", linewidth=0.7, zorder=6,
        )
ax_distribution.axhline(1.5, color="white", linewidth=2.0)
ax_distribution.axvline(0, color="#666666", linestyle="--", linewidth=1.0)
ax_distribution.set_title("Inferred proteins across NK subsets", loc="left", fontweight="bold")
ax_distribution.set_xlabel("Inferred protein z-score")
ax_distribution.set_ylabel("")
ax_distribution.grid(axis="x", color="white", linewidth=0.9)
ax_distribution.set_axisbelow(True)
ax_distribution.spines["top"].set_visible(False)
ax_distribution.spines["right"].set_visible(False)
ax_distribution.legend(
    title="", frameon=False, loc="lower center",
    bbox_to_anchor=(0.5, -0.22), ncol=2, fontsize=9.5,
)


fig.subplots_adjust(left=0.15, right=0.98, top=0.90, bottom=0.24)
save_figure(fig, "Fig2i_inferred_proteins_across_NK_subsets")
plt.show()

## NK paired-sample validation heatmap

In [ ]:
fig, ax_heatmap = plt.subplots(figsize=(8.3, 4.2))
max_abs = np.nanmax(np.abs(delta_display.to_numpy()))
sns.heatmap(
    delta_display, cmap="RdBu_r", center=0, vmin=-max_abs, vmax=max_abs,
    annot=True, fmt=".2f", annot_kws={"fontsize": 7.8},
    linewidths=0.7, linecolor="white",
    cbar_kws={"label": "Paired Δ z-score", "shrink": 0.76, "pad": 0.02},
    ax=ax_heatmap,
)
ax_heatmap.axhline(2, color="white", linewidth=2.5)
ax_heatmap.axvline(delta_matrix.shape[1], color="#444444", linewidth=1.6)
q_labels = []
for target in TARGET_PROTEINS:
    row = nk_contrasts.loc[nk_contrasts["protein"].eq(target)].iloc[0]
    q_value = row["paired_t_fdr"]
    q_text = "q<0.001" if q_value < 0.001 else f"q={q_value:.3f}"
    q_labels.append(f"{target}   {q_text}")
ax_heatmap.set_yticklabels(q_labels, rotation=0, fontsize=9.5)
ax_heatmap.set_xticklabels(ax_heatmap.get_xticklabels(), rotation=42, ha="right", fontsize=8.5)
ax_heatmap.set_title("Paired sample differences: CD16++ − CD56hi/CD16lo", loc="left", fontweight="bold")
ax_heatmap.set_xlabel("Biological sample")
ax_heatmap.set_ylabel("")


fig.tight_layout()
save_figure(fig, "NK_subtype_paired_sample_heatmap")
plt.show()

## Release temporary files

Keep the final tables, figures and trained models; delete this notebook’s temporary plotting/JIT caches.

In [ ]:
plt.close("all")
plot_cache.cleanup()
for key, previous in old_cache_settings.items():
    if previous is None:
        os.environ.pop(key, None)
    else:
        os.environ[key] = previous
print("Temporary caches removed.")